# Intelligent Surface Quality Analytics in Precision Machining
### Predictive Modeling for AISI H13 Tool Steel — CNC Turning Operations

**RAADHEY RAADHEY**

Name: **PRAGYA MEDHATIYA**  
Roll Number: **2401077**  
Branch: **ME** (CSE Minor)

---

In high-precision manufacturing, AISI H13 tool steel is a critical material utilized for high-pressure die casting and forging applications owing to its exceptional hot hardness (~52 HRC) and thermal fatigue resistance. However, machining H13 at production scale introduces a fundamental conflict: the same metallurgical properties that make the alloy industrially valuable — dense carbide networks, high chromium and molybdenum content — also make it notoriously difficult to cut. Minor fluctuations in feed rate, depth of cut, or progressive tool wear can trigger immediate degradation in surface topography, producing out-of-tolerance parts that must be scrapped at significant financial and energy cost.

This notebook develops a full-stack intelligent analytics pipeline that transitions surface quality control from a reactive, post-process measurement paradigm to a predictive, pre-process forecasting system. The pipeline spans raw data ingestion through advanced ensemble modeling, and is designed to act as an automated Virtual Quality Inspector on the shop floor.

**Dataset:** CNC Turning — Roughness, Forces and Tool Wear (Kaggle)  
**Source:** Canal, A. D. & Borille, A. V., Competence Center in Manufacturing (CCM), Aeronautics Institute of Technology (ITA), Brazil.  
**Material:** AISI H13 tool steel (~200 HV), machined on a ROMI E280 CNC turning center under cutting fluid conditions.

## 1 · Problem Statement — The Inefficiency of Post-Process Inspection

Traditional quality assurance in CNC turning relies entirely on offline measurement. After a part is machined, it is removed from the chuck, transferred to a profilometer station, and inspected. This reactive workflow introduces several compounding inefficiencies:

- **Delayed Detection:** If the arithmetic mean roughness Ra exceeds the engineering tolerance (typically 0.8–1.6 µm for finish turning), the part is already complete and must be scrapped or reworked — both of which consume additional machine time and operator labor.
- **Tool Uncertainty:** Operators replace cutting inserts based on conservative time-based schedules rather than actual flank wear state, leading to either premature tool waste (discarding inserts with remaining useful life) or unexpected surface failures (using worn inserts past their capability limit).
- **Energy and Material Waste:** Each rejected H13 part represents the total loss of the electrical energy consumed by the spindle drive and feed servos, the cutting fluid pumped through the operation, and the raw billet material itself.

The objective of this project is to construct a predictive regression model that ingests controllable machining parameters (cutting speed, feed rate, depth of cut, tool condition) along with measured cutting forces, and outputs a high-fidelity forecast of the resulting surface roughness Ra — before the part ever reaches the inspection station.

## 2 · Technical Architecture — The Data-to-Decision Pipeline

The analytical framework is organized into six integrated layers, each designed to progressively refine raw experimental data into actionable manufacturing intelligence:

1. **Data Ingestion Layer:** Three CSV files from controlled CNC turning experiments — two primary experiment phases (Exp1, Exp2) and one preparatory tool-conditioning phase (Prep) — are loaded and merged into a unified manufacturing matrix.
2. **Exploratory Data Analysis Layer:** Systematic statistical profiling, distribution mapping, correlation analysis, and physics-driven investigation of parameter interactions to understand the data-generating process before any modeling.
3. **Preprocessing Layer:** Missing value imputation, duplicate elimination, categorical encoding, IQR-based outlier capping, and standardization to produce a numerically stable feature matrix.
4. **Feature Engineering Layer:** Domain-informed construction of derived features — polynomial terms, interaction products, physical ratios, and force decompositions — that encode the known physics of the metal-cutting process into the feature space.
5. **Modeling Layer:** A comprehensive model evaluation spanning linear baselines, regularized regressors, tree-based ensembles, kernel machines, and a stacking meta-learner — each rigorously validated through stratified cross-validation.
6. **Interpretation & Deployment Layer:** Feature importance analysis via permutation testing, residual diagnostics, and model serialization for factory-floor integration.

## 3 · Technologies Used — The Engineering Stack

| Layer | Technology | Purpose |
|---|---|---|
| Core Language | Python 3.x | Data processing, algorithm development, pipeline orchestration |
| Data Handling | Pandas, NumPy | Tabular manipulation, vectorized numerical computation |
| Statistical Analysis | SciPy | Hypothesis testing (Shapiro-Wilk, Kruskal-Wallis, Spearman), distribution fitting |
| Visualization | Matplotlib, Seaborn | Publication-grade statistical graphics, heatmaps, violin plots |
| Machine Learning | Scikit-Learn | Regression algorithms, cross-validation, hyperparameter search, feature importance |
| Model Persistence | Joblib | Serialization of trained estimators for deployment |

## 4 · Dataset Description

The dataset originates from a controlled CNC turning campaign on AISI H13 steel workpieces (mean hardness ~200 HV) conducted at ITA's CCM laboratory. The experiments used a ROMI E280 CNC turning center with Mitsubishi CNMG120408 carbide inserts under wet cutting conditions. Surface roughness was measured with a Mitutoyo portable profilometer; cutting forces were acquired with a Kistler Type 9265B piezoelectric dynamometer.

The raw data spans three files and 824 total records across two experimental phases and one preparatory conditioning phase:

**Controllable Inputs (Machine Settings):**
- `vc` — Cutting speed [m/min]: {310, 350, 390}. Governs the peripheral velocity of the workpiece relative to the tool tip; directly influences thermal load and chip morphology.
- `f` — Feed rate [mm/rev]: {0.07, 0.09, 0.10, 0.11, 0.13}. Controls the axial advance per revolution; the single most influential parameter on theoretical surface roughness (Ra ∝ f²/8r).
- `ap` — Depth of cut [mm]: {0.25, 0.50, 0.80}. Sets the radial engagement of the tool; affects chip cross-section area and resultant cutting force magnitude.
- `TCond` — Tool condition (flank wear) [mm]: {0.0, 0.1, 0.3} in Exp1/Exp2; continuous in Prep.

**Measured Responses (Physical Outputs):**
- `Ra` — Arithmetic mean roughness [µm]: The primary target.
- `Rz`, `Rsk`, `Rku`, `RSm`, `Rt` — Supplementary roughness parameters.
- `Fx`, `Fy`, `Fz` — Orthogonal cutting force components [N]. `F` — Resultant force magnitude.

**Administrative Metadata:**
- `Run_ID`, `Experiment`, `Replica`, `Group`, `Subgroup`, `Position` — Organizational labels; dropped during preprocessing.
- `Init_diameter`, `Final_diameter` — Workpiece geometry before/after cut.
- `Machined_length`, `CTime` — Cut length and cycle time.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, kruskal, spearmanr, pearsonr
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 150
print('Environment ready.')

## 5 · Data Ingestion

Three separate CSV files capture distinct experimental phases. Exp1 (324 records) and Exp2 (288 records) contain the primary factorial experiments under controlled cutting conditions. Prep (212 records) captures the preparatory conditioning passes used to establish initial tool wear states — it includes a continuous tool wear progression rather than the discrete {0.0, 0.1, 0.3} levels in the main experiments, and lacks force measurements.

Merging Exp1 and Exp2 into a single dataframe doubles the effective sample size and exposes the model to a broader range of cutting condition combinations. Prep is kept separate for a dedicated tool wear degradation analysis.

In [ ]:
df1 = pd.read_csv('Exp1.csv')
df2 = pd.read_csv('Exp2.csv')
prep = pd.read_csv('Prep.csv')

df = pd.concat([df1, df2], ignore_index=True)

print(f'Exp1:     {df1.shape[0]} records x {df1.shape[1]} columns')
print(f'Exp2:     {df2.shape[0]} records x {df2.shape[1]} columns')
print(f'Prep:     {prep.shape[0]} records x {prep.shape[1]} columns')
print(f'Combined: {df.shape[0]} records x {df.shape[1]} columns')
print(f'\nColumn list: {df.columns.tolist()}')

### 5.1 · Initial Structural Audit

Before any transformation, a raw inspection of the first and last rows, data types, and memory footprint confirms that the CSV parsing preserved the expected schema. This sanity check catches silent type coercions early in the pipeline.

In [ ]:
print(df.head().to_string())
print()
print(df.tail().to_string())
print()
df.info(memory_usage='deep')

### 5.2 · Descriptive Statistics — First Statistical Fingerprint

The five-number summary plus mean and standard deviation for every numeric column establishes the baseline statistical profile of the dataset.

In [ ]:
print(df.describe().round(3).to_string())

## 6 · Data Quality Assessment

### 6.1 · Missing Value Cartography

A systematic null audit across every column reveals the extent and pattern of missing data. In experimental manufacturing datasets, missingness is rarely random — sensor dropouts tend to cluster around extreme cutting conditions where vibration disrupts the data acquisition chain.

In [ ]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)

null_report = pd.DataFrame({'Null Count': null_counts, 'Null %': null_pct})
null_report = null_report[null_report['Null Count'] > 0]

if len(null_report) > 0:
    print('Columns with missing values:')
    print(null_report)
else:
    print('No null values detected in any column.')

na_strings = df.isin(['na', 'NA', 'N/A', 'n/a', '']).sum()
na_strings = na_strings[na_strings > 0]
if len(na_strings) > 0:
    print('\nString sentinel values detected:')
    print(na_strings)

### 6.2 · Data Cleaning Pipeline

The cleaning pipeline handles three distinct issues in sequence:
1. **String sentinel replacement:** The raw CSVs use the literal string 'na' instead of proper NaN.
2. **Type coercion:** Columns that should be numeric but were read as object dtype are force-converted.
3. **Administrative column removal:** Columns serving purely organizational purposes are dropped.

In [ ]:
df = df.replace('na', np.nan)

for col in df.columns:
    if df[col].dtype == 'object':
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            pass

admin_cols = ['Run_ID', 'Experiment', 'Replica', 'Group', 'Subgroup', 'R_measurement']
df = df.drop(columns=[c for c in admin_cols if c in df.columns])
df = df.dropna(axis=1, how='all')

print(f'Shape after cleanup: {df.shape}')
print(f'Remaining columns: {df.columns.tolist()}')
print(f'\nData types:')
print(df.dtypes)

### 6.3 · Missing Value Imputation

For numeric columns, median imputation is preferred over mean imputation because cutting force and roughness distributions in machining data tend to be right-skewed — the median is robust to the extreme values produced by heavy cuts or worn tools.

In [ ]:
from sklearn.impute import SimpleImputer

num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

if len(num_cols) > 0:
    num_imputer = SimpleImputer(strategy='median')
    df[num_cols] = num_imputer.fit_transform(df[num_cols])

if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print(f'Remaining nulls: {df.isnull().sum().sum()}')

### 6.4 · Duplicate Record Elimination

In multi-replica experimental designs, exact duplicate rows can arise from data entry errors. Removing true duplicates prevents the model from seeing the same physical measurement twice, which would artificially inflate training accuracy.

In [ ]:
dup_count = df.duplicated().sum()
print(f'Exact duplicate rows found: {dup_count}')
df = df.drop_duplicates(ignore_index=True)
print(f'Shape after dedup: {df.shape}')

### 6.5 · Categorical Feature Encoding

The Position column encodes the circumferential measurement location on the workpiece (a through j). Since these positions have no inherent ordinal ranking, one-hot encoding with drop='first' is used to avoid the dummy variable trap.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cat_cols = df.select_dtypes(include=['object']).columns.tolist()

if len(cat_cols) > 0:
    print(f'Categorical columns to encode: {cat_cols}')
    transformer = ColumnTransformer(
        transformers=[('cat', OneHotEncoder(sparse_output=False, drop='first'), cat_cols)],
        remainder='passthrough',
        verbose_feature_names_out=False
    ).set_output(transform='pandas')
    df = transformer.fit_transform(df)
    print(f'Shape after encoding: {df.shape}')
else:
    print('No categorical columns remaining.')

print(f'Final column count: {len(df.columns)}')

## 7 · Exploratory Data Analysis — Mapping the Physics

### 7.1 · Target Distribution Analysis

The distribution of the target variable Ra is the single most important diagnostic before model selection. A heavily skewed or multimodal distribution may require a log-transform; a roughly normal distribution suggests standard regression approaches will work well.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df['Ra'], bins=35, color='steelblue', edgecolor='white', alpha=0.85, density=True)
kde_x = np.linspace(df['Ra'].min(), df['Ra'].max(), 300)
kde = stats.gaussian_kde(df['Ra'])
axes[0].plot(kde_x, kde(kde_x), color='firebrick', lw=2)
axes[0].set_xlabel('Surface Roughness Ra')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of Ra')
axes[0].axvline(df['Ra'].mean(), color='black', ls='--', lw=1.2, label=f"Mean = {df['Ra'].mean():.3f}")
axes[0].axvline(df['Ra'].median(), color='orange', ls='--', lw=1.2, label=f"Median = {df['Ra'].median():.3f}")
axes[0].legend(fontsize=9)

stats.probplot(df['Ra'], dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot — Ra vs. Normal Distribution')
axes[1].get_lines()[0].set_markerfacecolor('steelblue')
axes[1].get_lines()[0].set_markersize(4)

bp = axes[2].boxplot(df['Ra'], vert=True, patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][0].set_alpha(0.7)
axes[2].set_ylabel('Ra')
axes[2].set_title('Box Plot — Ra Spread and Outliers')

plt.tight_layout()
plt.show()

stat_sw, p_sw = shapiro(df['Ra'].sample(min(500, len(df['Ra'])), random_state=SEED))
skewness = df['Ra'].skew()
kurt = df['Ra'].kurtosis()
print(f'Ra — Skewness: {skewness:.3f} | Kurtosis: {kurt:.3f}')
print(f'Shapiro-Wilk test: W={stat_sw:.4f}, p={p_sw:.4f}')
print(f'  -> {"Reject" if p_sw < 0.05 else "Fail to reject"} normality at alpha=0.05')

### 7.2 · Univariate Parameter Effects on Surface Roughness

Each machining parameter exerts a distinct physical influence on the surface generation mechanism. Violin plots expose the full conditional distribution of Ra at each factor level — revealing shifts in central tendency and changes in variance and modality.

**Physical expectations:**
- Feed rate (f) should show the strongest positive effect on Ra, since theoretical roughness scales quadratically with feed.
- Cutting speed (vc) typically has a non-monotonic effect.
- Depth of cut (ap) has a secondary effect through its influence on cutting force magnitude.
- Tool condition (TCond) captures progressive flank wear.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

params = [('f', 'Feed Rate f [mm/rev]'),
          ('vc', 'Cutting Speed vc [m/min]'),
          ('ap', 'Depth of Cut ap [mm]'),
          ('TCond', 'Tool Condition TCond [mm wear]')]

colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for idx, (col, label) in enumerate(params):
    ax = axes[idx // 2][idx % 2]
    groups = sorted(df[col].unique())
    data_groups = [df[df[col] == g]['Ra'].values for g in groups]

    vp = ax.violinplot(data_groups, positions=range(len(groups)), showmeans=True, showmedians=True)
    for body in vp['bodies']:
        body.set_facecolor(colors[idx])
        body.set_alpha(0.6)
    vp['cmeans'].set_color('black')
    vp['cmedians'].set_color('orange')

    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels([str(g) for g in groups])
    ax.set_xlabel(label)
    ax.set_ylabel('Ra')
    ax.set_title(f'Ra Distribution by {col}')

    for i, d in enumerate(data_groups):
        jitter = np.random.normal(0, 0.04, size=len(d))
        ax.scatter(np.full_like(d, i) + jitter, d, alpha=0.15, s=8, color='black')

plt.tight_layout()
plt.show()

print('--- Kruskal-Wallis H-test: Ra across factor levels ---')
for col, label in params:
    groups_data = [df[df[col] == g]['Ra'].values for g in sorted(df[col].unique())]
    h_stat, p_val = kruskal(*groups_data)
    print(f'  {col:6s}: H={h_stat:8.2f}, p={p_val:.2e}  -> {"Significant" if p_val < 0.05 else "Not significant"}')

### 7.3 · Correlation Structure — Mapping Variable Interdependencies

The Pearson correlation heatmap reveals the linear association structure across the entire feature space. The lower triangle is masked to eliminate redundancy, and a diverging colormap centered at zero highlights the polarity of each relationship.

In [ ]:
corr = df.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.75, 'label': 'Pearson r'}, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Pearson Correlation Matrix', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

ra_corr = corr['Ra'].drop('Ra').sort_values(ascending=False)
print('--- Correlation with Ra (sorted) ---')
for feat, val in ra_corr.items():
    if pd.notna(val):
        bar = '#' * int(abs(val) * 30)
        sign = '+' if val > 0 else '-'
        print(f'  {feat:20s}  {sign}{abs(val):.3f}  {bar}')

### 7.4 · Two-Factor Interaction Effects

Surface roughness in metal cutting is not governed by isolated parameters — it emerges from their simultaneous interaction. These interaction plots are the empirical equivalent of the interaction terms in a factorial ANOVA and directly inform which polynomial cross-terms to include in feature engineering.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for vc_val in sorted(df['vc'].unique()):
    sub = df[df['vc'] == vc_val].groupby('f')['Ra'].mean()
    axes[0].plot(sub.index, sub.values, 'o-', label=f'vc={vc_val}', lw=2, markersize=6)
axes[0].set_xlabel('Feed Rate f [mm/rev]')
axes[0].set_ylabel('Mean Ra')
axes[0].set_title('Interaction: Feed Rate x Cutting Speed')
axes[0].legend(title='vc [m/min]')

for ap_val in sorted(df['ap'].unique()):
    sub = df[df['ap'] == ap_val].groupby('f')['Ra'].mean()
    axes[1].plot(sub.index, sub.values, 's-', label=f'ap={ap_val}', lw=2, markersize=6)
axes[1].set_xlabel('Feed Rate f [mm/rev]')
axes[1].set_ylabel('Mean Ra')
axes[1].set_title('Interaction: Feed Rate x Depth of Cut')
axes[1].legend(title='ap [mm]')

for tc_val in sorted(df['TCond'].unique()):
    sub = df[df['TCond'] == tc_val].groupby('f')['Ra'].mean()
    axes[2].plot(sub.index, sub.values, '^-', label=f'TCond={tc_val}', lw=2, markersize=6)
axes[2].set_xlabel('Feed Rate f [mm/rev]')
axes[2].set_ylabel('Mean Ra')
axes[2].set_title('Interaction: Feed Rate x Tool Condition')
axes[2].legend(title='TCond [mm]')

plt.tight_layout()
plt.show()

### 7.5 · Cutting Force Decomposition and Ra Relationship

Fx (feed force), Fy (radial/thrust force), and Fz (tangential/main cutting force) carry rich information about the metal-cutting mechanics. The Fy component, acting perpendicular to the machined surface, directly influences surface generation through tool deflection.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
force_cols = ['Fx', 'Fy', 'Fz']
force_labels = ['Feed Force Fx [N]', 'Radial Force Fy [N]', 'Tangential Force Fz [N]']

for i, (fc, fl) in enumerate(zip(force_cols, force_labels)):
    scatter = axes[i].scatter(df[fc], df['Ra'], c=df['f'], cmap='viridis',
                              alpha=0.6, s=20, edgecolors='white', linewidth=0.3)
    z = np.polyfit(df[fc], df['Ra'], 1)
    p_line = np.poly1d(z)
    x_line = np.linspace(df[fc].min(), df[fc].max(), 100)
    axes[i].plot(x_line, p_line(x_line), 'r--', lw=1.5, alpha=0.8)
    r_val, p_val = pearsonr(df[fc], df['Ra'])
    axes[i].set_xlabel(fl)
    axes[i].set_ylabel('Ra')
    axes[i].set_title(f'{fc} vs Ra  (r = {r_val:.3f})')

plt.colorbar(scatter, ax=axes[-1], label='Feed Rate f', shrink=0.8)
plt.tight_layout()
plt.show()

print('--- Spearman rho with Ra ---')
for fc in ['Fx', 'Fy', 'Fz', 'F']:
    rho, p = spearmanr(df[fc], df['Ra'])
    print(f'  {fc:3s}: rho = {rho:+.3f}  (p = {p:.2e})')

### 7.6 · Tool Wear Degradation Trajectory (Prep Data)

The Prep dataset captures the preparatory conditioning phase where tools were progressively worn from a fresh state. Unlike the main experiments which sample only three discrete wear levels, Prep provides a near-continuous wear progression.

In [ ]:
prep_clean = prep.copy()
prep_clean = prep_clean.replace('na', np.nan)
for col in prep_clean.columns:
    if prep_clean[col].dtype == 'object':
        try:
            prep_clean[col] = pd.to_numeric(prep_clean[col])
        except (ValueError, TypeError):
            pass

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(prep_clean['TCond'], prep_clean['Ra'], alpha=0.5, s=25, color='steelblue',
                edgecolors='white', linewidth=0.3)
sorted_prep = prep_clean[['TCond', 'Ra']].dropna().sort_values('TCond')
window = max(5, len(sorted_prep) // 15)
rolling_mean = sorted_prep.rolling(window, center=True, on='TCond')['Ra'].mean()
axes[0].plot(sorted_prep['TCond'], rolling_mean, color='firebrick', lw=2.5, label=f'Rolling mean (w={window})')
axes[0].set_xlabel('TCond [mm flank wear]')
axes[0].set_ylabel('Ra')
axes[0].set_title('Tool Wear Degradation Trajectory')
axes[0].legend()

axes[1].scatter(prep_clean['TCond'], prep_clean['Rz'], alpha=0.5, s=25, color='darkorange',
                edgecolors='white', linewidth=0.3)
axes[1].set_xlabel('TCond [mm]')
axes[1].set_ylabel('Rz')
axes[1].set_title('Rz vs. Tool Wear')

axes[2].scatter(prep_clean['TCond'], prep_clean['Rsk'], alpha=0.4, s=20, label='Rsk', color='#55A868')
ax2 = axes[2].twinx()
ax2.scatter(prep_clean['TCond'], prep_clean['Rku'], alpha=0.4, s=20, label='Rku', color='#C44E52')
axes[2].set_xlabel('TCond [mm]')
axes[2].set_ylabel('Rsk')
ax2.set_ylabel('Rku')
axes[2].set_title('Profile Shape vs. Tool Wear')
lines1, labels1 = axes[2].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[2].legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

rho_prep, p_prep = spearmanr(prep_clean['TCond'].dropna(), prep_clean.loc[prep_clean['TCond'].notna(), 'Ra'])
print(f'Prep data — Spearman rho(TCond, Ra) = {rho_prep:+.3f}  (p = {p_prep:.2e})')

### 7.7 · Multi-Roughness Parameter Covariance

The six roughness parameters describe different aspects of the same physical surface profile. Understanding their covariance structure reveals which parameters carry independent information versus which are near-redundant.

In [ ]:
roughness_cols = ['Ra', 'Rz', 'Rsk', 'Rku', 'RSm', 'Rt']
roughness_sub = df[roughness_cols]

g = sns.pairplot(roughness_sub, diag_kind='kde',
                 plot_kws={'alpha': 0.3, 's': 15, 'edgecolor': 'white', 'linewidth': 0.2},
                 diag_kws={'color': 'steelblue', 'lw': 2})
g.figure.suptitle('Roughness Parameter Pairwise Relationships', y=1.01, fontsize=14)
plt.show()

print('--- Inter-roughness Pearson correlations with Ra ---')
for col in ['Rz', 'Rsk', 'Rku', 'RSm', 'Rt']:
    r, p = pearsonr(df['Ra'], df[col])
    print(f'  Ra <-> {col:4s}: r = {r:+.3f}  (p = {p:.2e})')

## 8 · Preprocessing Pipeline

### 8.1 · Outlier Detection and Capping (Winsorization)

Rather than deleting valuable records, we cap extreme values at the IQR fence boundaries. This preserves the sample size while absorbing the mathematical leverage that outliers exert on regression coefficients.

In [ ]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].set_title('Before Outlier Capping')
df[['Ra', 'Rz', 'Rt', 'Fx', 'Fy', 'Fz']].boxplot(ax=axes[0], vert=True)
axes[0].tick_params(axis='x', rotation=45)

outlier_report = {}
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_clipped = ((df[col] < lower) | (df[col] > upper)).sum()
    if n_clipped > 0:
        outlier_report[col] = n_clipped
    df[col] = df[col].clip(lower, upper)

axes[1].set_title('After Outlier Capping (IQR Winsorization)')
df[['Ra', 'Rz', 'Rt', 'Fx', 'Fy', 'Fz']].boxplot(ax=axes[1], vert=True)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('--- Outlier Capping Report ---')
for col, n in sorted(outlier_report.items(), key=lambda x: -x[1]):
    print(f'  {col:20s}: {n:3d} values capped ({n/len(df)*100:.1f}%)')

### 8.2 · Feature Standardization

StandardScaler transforms each feature to zero mean and unit variance. This is critical for regularized models (Ridge, Lasso), SVR, and gradient-based optimizers.

In [ ]:
from sklearn.preprocessing import StandardScaler

df_unscaled = df.copy()
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print('Post-scaling verification (should be ~0 mean, ~1 std):')
print(df[num_cols].describe().loc[['mean', 'std']].round(4).to_string())

### 8.3 · Preprocessed Data Export

Exporting the fully preprocessed manufacturing matrix into Excel format ensures the cleaned data is permanently preserved for downstream pipelines.

In [ ]:
df.to_excel('Preprocessed_AISI_H13_Turning_Data.xlsx', index=False)
print('Exported: Preprocessed_AISI_H13_Turning_Data.xlsx')

## 9 · Feature Engineering — Encoding the Physics

Raw machining parameters capture only first-order effects. The actual surface generation mechanism involves complex non-linear interactions:
- **Ra proportional to f squared/8r** (theoretical kinematic roughness) — motivates polynomial terms in f.
- **Chip cross-section area = ap x f** — governs cutting force magnitude and heat generation.
- **Material removal rate = vc x f x ap** — determines energy input rate and thermal load.
- **Specific cutting energy = F / (ap x f x vc)** — normalized measure of cutting efficiency.

By explicitly constructing these physics-informed features, we reduce the burden on the learner to discover these relationships from raw data alone.

In [ ]:
df_eng = df_unscaled.copy()

# Polynomial terms
df_eng['f_squared'] = df_eng['f'] ** 2
df_eng['vc_squared'] = df_eng['vc'] ** 2
df_eng['ap_squared'] = df_eng['ap'] ** 2

# Two-factor interactions
df_eng['vc_x_f'] = df_eng['vc'] * df_eng['f']
df_eng['vc_x_ap'] = df_eng['vc'] * df_eng['ap']
df_eng['f_x_ap'] = df_eng['f'] * df_eng['ap']
df_eng['f_x_TCond'] = df_eng['f'] * df_eng['TCond']
df_eng['vc_x_TCond'] = df_eng['vc'] * df_eng['TCond']

# Physics-derived features
df_eng['chip_area'] = df_eng['ap'] * df_eng['f']
df_eng['MRR'] = df_eng['vc'] * df_eng['f'] * df_eng['ap']
df_eng['specific_energy'] = df_eng['F'] / (df_eng['MRR'] + 1e-9)
df_eng['Fy_to_Fz_ratio'] = df_eng['Fy'] / (df_eng['Fz'] + 1e-9)
df_eng['F_per_chip_area'] = df_eng['F'] / (df_eng['chip_area'] + 1e-9)

# Diameter reduction
if 'Init_diameter' in df_eng.columns and 'Final_diameter' in df_eng.columns:
    df_eng['diameter_reduction'] = df_eng['Init_diameter'] - df_eng['Final_diameter']

# Log transforms for skewed features
df_eng['log_F'] = np.log1p(df_eng['F'])

print(f'Engineered feature count: {len(df_eng.columns)}')
new_feats = [c for c in df_eng.columns if c not in df_unscaled.columns]
print(f'New features ({len(new_feats)}): {new_feats}')

### 9.1 · Feature-Target Correlation Audit

A ranked correlation audit identifies which engineered features show the strongest association with Ra.

In [ ]:
exclude_targets = ['Ra', 'Rz', 'Rsk', 'Rku', 'RSm', 'Rt']
feature_candidates = [c for c in df_eng.select_dtypes(include=[np.number]).columns if c not in exclude_targets]

corr_with_ra = df_eng[feature_candidates + ['Ra']].corr()['Ra'].drop('Ra').sort_values(ascending=False)
corr_with_ra = corr_with_ra.dropna()

fig, ax = plt.subplots(figsize=(10, max(6, len(corr_with_ra) * 0.3)))
colors = ['#4C72B0' if v > 0 else '#C44E52' for v in corr_with_ra.values]
ax.barh(range(len(corr_with_ra)), corr_with_ra.values, color=colors, edgecolor='white', height=0.7)
ax.set_yticks(range(len(corr_with_ra)))
ax.set_yticklabels(corr_with_ra.index, fontsize=8)
ax.set_xlabel('Pearson Correlation with Ra')
ax.set_title('Feature-Target Correlation Ranking')
ax.axvline(x=0, color='black', lw=0.8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 10 features by |correlation| with Ra:')
top10 = corr_with_ra.abs().sort_values(ascending=False).head(10)
for feat in top10.index:
    print(f'  {feat:25s}: {corr_with_ra[feat]:+.3f}')

## 10 · Model Development — Train/Test Split

The dataset is partitioned into 80% training and 20% holdout test sets. The feature matrix X excludes all roughness response variables (since those are measured simultaneously with Ra and would leak the target). Only features available before the cut is completed are retained.

In [ ]:
from sklearn.model_selection import train_test_split

target = 'Ra'
drop_cols = ['Ra', 'Rz', 'Rsk', 'Rku', 'RSm', 'Rt']
admin_leftovers = ['Run_ID', 'Experiment', 'Replica', 'Group', 'Subgroup',
                   'R_measurement', 'CTime', 'Machined_length']
drop_cols += [c for c in admin_leftovers if c in df_eng.columns]

X = df_eng.drop(columns=[c for c in drop_cols if c in df_eng.columns], errors='ignore')
X = X.select_dtypes(include=[np.number])
y = df_eng[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)

print(f'Feature matrix shape: {X.shape}')
print(f'Training set:  {X_train.shape[0]} samples')
print(f'Holdout test:  {X_test.shape[0]} samples')
print(f'\nFeatures used ({X.shape[1]}):')
for i, col in enumerate(X.columns):
    print(f'  {i+1:2d}. {col}')

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_model = StandardScaler()
X_train_sc = pd.DataFrame(scaler_model.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_sc = pd.DataFrame(scaler_model.transform(X_test), columns=X_test.columns, index=X_test.index)

print('Scaling applied. Train/Test shapes:', X_train_sc.shape, X_test_sc.shape)

## 11 · Baseline Model Evaluation

A comprehensive sweep across nine regression architectures establishes the performance landscape before hyperparameter tuning.

| Model | Inductive Bias |
|---|---|
| Linear Regression | Strictly linear input-output mapping |
| Ridge Regression | Linear + L2 penalty on coefficients |
| Lasso Regression | Linear + L1 penalty (sparsity) |
| ElasticNet | L1 + L2 hybrid |
| Decision Tree | Recursive axis-aligned partitioning |
| Random Forest | Bagged ensemble of decorrelated trees |
| Gradient Boosting | Sequential residual-correcting trees |
| SVR (RBF kernel) | Implicit infinite-dimensional feature mapping |
| K-Nearest Neighbors | Instance-based local averaging |

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import time

models = {
    'Linear Regression':  LinearRegression(),
    'Ridge Regression':   Ridge(alpha=1.0),
    'Lasso Regression':   Lasso(alpha=0.01, max_iter=10000),
    'ElasticNet':         ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
    'Decision Tree':      DecisionTreeRegressor(random_state=SEED),
    'Random Forest':      RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=200, random_state=SEED),
    'SVR (RBF)':          SVR(kernel='rbf', C=10, gamma='scale'),
    'KNN':                KNeighborsRegressor(n_neighbors=5),
}

baseline_results = {}

print(f'{"Model":30s} {"CV R2":>10s} {"CV RMSE":>10s} {"Test R2":>10s} {"Test RMSE":>10s} {"Time":>8s}')
print('-' * 78)

for name, model in models.items():
    t0 = time.time()
    cv_r2 = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2')
    cv_rmse = -cross_val_score(model, X_train_sc, y_train, cv=5, scoring='neg_root_mean_squared_error')
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    test_r2 = r2_score(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    elapsed = time.time() - t0

    baseline_results[name] = {
        'CV R2 mean': cv_r2.mean(), 'CV R2 std': cv_r2.std(),
        'CV RMSE': cv_rmse.mean(), 'Test R2': test_r2, 'Test RMSE': test_rmse, 'model': model
    }
    print(f'{name:30s} {cv_r2.mean():10.4f} {cv_rmse.mean():10.4f} {test_r2:10.4f} {test_rmse:10.4f} {elapsed:7.1f}s')

### 11.1 · Baseline Performance Comparison

The bar chart provides a visual ranking of all baseline models on the holdout test set. The dashed line marks R2=0.90 — a common industry benchmark for manufacturing process models.

In [ ]:
results_df = pd.DataFrame({
    name: {'Test R2': v['Test R2'], 'Test RMSE': v['Test RMSE']}
    for name, v in baseline_results.items()
}).T.sort_values('Test R2', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

results_df['Test R2'].plot.barh(ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(x=0.90, color='firebrick', ls='--', lw=1.5, label='R2 = 0.90')
axes[0].set_xlabel('R2 Score')
axes[0].set_title('Holdout Test R2 — All Baselines')
axes[0].legend()

results_df['Test RMSE'].plot.barh(ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_xlabel('RMSE')
axes[1].set_title('Holdout Test RMSE — All Baselines')

plt.tight_layout()
plt.show()

## 12 · Hyperparameter Tuning via Grid Search

The top-performing baseline models are refined through exhaustive grid search with 5-fold cross-validation.

In [ ]:
from sklearn.model_selection import GridSearchCV

dt_params = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
dt_grid = GridSearchCV(DecisionTreeRegressor(random_state=SEED), dt_params,
                       cv=5, scoring='r2', n_jobs=-1)
dt_grid.fit(X_train_sc, y_train)

rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 0.5, 0.8]
}
rf_grid = GridSearchCV(RandomForestRegressor(random_state=SEED, n_jobs=-1), rf_params,
                       cv=5, scoring='r2', n_jobs=-1)
rf_grid.fit(X_train_sc, y_train)

gb_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}
gb_grid = GridSearchCV(GradientBoostingRegressor(random_state=SEED), gb_params,
                       cv=5, scoring='r2', n_jobs=-1)
gb_grid.fit(X_train_sc, y_train)

print('--- Best Hyperparameters ---')
print(f'Decision Tree:      {dt_grid.best_params_}  ->  CV R2 = {dt_grid.best_score_:.4f}')
print(f'Random Forest:      {rf_grid.best_params_}  ->  CV R2 = {rf_grid.best_score_:.4f}')
print(f'Gradient Boosting:  {gb_grid.best_params_}  ->  CV R2 = {gb_grid.best_score_:.4f}')

### 12.1 · Tuned Model Evaluation on Holdout Set

In [ ]:
best_dt = dt_grid.best_estimator_
best_rf = rf_grid.best_estimator_
best_gb = gb_grid.best_estimator_

tuned_models = {
    'Tuned Decision Tree': best_dt,
    'Tuned Random Forest': best_rf,
    'Tuned Gradient Boosting': best_gb,
}

print(f'{"Model":30s} {"CV R2":>10s} {"Test R2":>10s} {"Test RMSE":>10s} {"Test MAE":>10s}')
print('-' * 72)

tuned_results = {}
for name, model in tuned_models.items():
    y_pred = model.predict(X_test_sc)
    test_r2 = r2_score(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae = mean_absolute_error(y_test, y_pred)
    cv_score = {'Tuned Decision Tree': dt_grid, 'Tuned Random Forest': rf_grid,
                'Tuned Gradient Boosting': gb_grid}[name].best_score_
    tuned_results[name] = {'CV R2': cv_score, 'Test R2': test_r2, 'Test RMSE': test_rmse,
                           'Test MAE': test_mae, 'model': model}
    print(f'{name:30s} {cv_score:10.4f} {test_r2:10.4f} {test_rmse:10.4f} {test_mae:10.4f}')

## 13 · Stacking Ensemble — Meta-Learner Architecture

A stacking ensemble combines the strengths of multiple model families by training a meta-learner (Ridge regression) on the cross-validated predictions of the base models. This two-layer architecture allows the meta-learner to learn the optimal weighting of each base model's predictions across different regions of the input space.

In [ ]:
from sklearn.ensemble import StackingRegressor

stacking = StackingRegressor(
    estimators=[
        ('rf', best_rf),
        ('gb', best_gb),
        ('ridge', Ridge(alpha=1.0)),
        ('knn', KNeighborsRegressor(n_neighbors=7)),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5, n_jobs=-1, passthrough=False
)

stacking.fit(X_train_sc, y_train)
y_pred_stack = stacking.predict(X_test_sc)

stack_r2 = r2_score(y_test, y_pred_stack)
stack_rmse = np.sqrt(mean_squared_error(y_test, y_pred_stack))
stack_mae = mean_absolute_error(y_test, y_pred_stack)

print(f'Stacking Ensemble — Test R2: {stack_r2:.4f}  |  RMSE: {stack_rmse:.4f}  |  MAE: {stack_mae:.4f}')

## 14 · Comprehensive Model Comparison

The final leaderboard aggregates all baseline, tuned, and ensemble models, sorted by holdout test R2.

In [ ]:
all_models = {}
for name, v in baseline_results.items():
    y_p = v['model'].predict(X_test_sc)
    all_models[name] = {'Test R2': r2_score(y_test, y_p),
                        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_p)),
                        'Test MAE': mean_absolute_error(y_test, y_p)}

for name, v in tuned_results.items():
    all_models[name] = {'Test R2': v['Test R2'], 'Test RMSE': v['Test RMSE'], 'Test MAE': v['Test MAE']}

all_models['Stacking Ensemble'] = {'Test R2': stack_r2, 'Test RMSE': stack_rmse, 'Test MAE': stack_mae}

leaderboard = pd.DataFrame(all_models).T.sort_values('Test R2', ascending=False)
print('\n======== COMPREHENSIVE MODEL LEADERBOARD ========')
print(leaderboard.to_string())
print('==================================================')

fig, ax = plt.subplots(figsize=(10, 7))
lb_sorted = leaderboard.sort_values('Test R2', ascending=True)
colors = ['#2ecc71' if r >= 0.95 else '#f39c12' if r >= 0.90 else '#e74c3c'
          for r in lb_sorted['Test R2']]
lb_sorted['Test R2'].plot.barh(ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('R2 Score')
ax.set_title('Final Model Leaderboard — Holdout Test R2')
ax.axvline(x=0.95, color='green', ls=':', lw=1, alpha=0.7, label='R2=0.95')
ax.axvline(x=0.90, color='orange', ls=':', lw=1, alpha=0.7, label='R2=0.90')
ax.legend()
plt.tight_layout()
plt.show()

## 15 · Feature Importance Analysis

### 15.1 · Permutation Importance (Model-Agnostic)

Permutation importance measures the decrease in model performance when a single feature's values are randomly shuffled, breaking its relationship with the target.

In [ ]:
from sklearn.inspection import permutation_importance

best_model_name = leaderboard.index[0]
print(f'Analyzing feature importance for: {best_model_name}')

if best_model_name == 'Stacking Ensemble':
    importance_model = stacking
elif best_model_name in tuned_results:
    importance_model = tuned_results[best_model_name]['model']
else:
    importance_model = baseline_results[best_model_name]['model']

perm_imp = permutation_importance(importance_model, X_test_sc, y_test,
                                   n_repeats=30, random_state=SEED, n_jobs=-1)

perm_df = pd.DataFrame({
    'Feature': X_test_sc.columns,
    'Importance Mean': perm_imp.importances_mean,
    'Importance Std': perm_imp.importances_std
}).sort_values('Importance Mean', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(perm_df) * 0.3)))
ax.barh(perm_df['Feature'], perm_df['Importance Mean'],
        xerr=perm_df['Importance Std'], color='steelblue', edgecolor='white', capsize=2)
ax.set_xlabel('Mean R2 Decrease')
ax.set_title(f'Permutation Feature Importance — {best_model_name}')
plt.tight_layout()
plt.show()

print('\nTop 10 most important features:')
for _, row in perm_df.sort_values('Importance Mean', ascending=False).head(10).iterrows():
    print(f"  {row['Feature']:25s}: {row['Importance Mean']:.4f} +/- {row['Importance Std']:.4f}")

### 15.2 · Impurity-Based Feature Importance (Random Forest)

In [ ]:
rf_imp = pd.Series(best_rf.feature_importances_, index=X_train_sc.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(rf_imp) * 0.3)))
rf_imp.plot.barh(ax=ax, color='darkorange', edgecolor='white')
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('Random Forest — Impurity-Based Feature Importance')
plt.tight_layout()
plt.show()

## 16 · Residual Diagnostics

### 16.1 · Actual vs. Predicted and Residual Analysis

Well-calibrated predictions produce points tightly clustered along the y=x diagonal, a random residual cloud centered at zero, and a symmetric residual histogram.

In [ ]:
if best_model_name == 'Stacking Ensemble':
    y_pred_best = y_pred_stack
elif best_model_name in tuned_results:
    y_pred_best = tuned_results[best_model_name]['model'].predict(X_test_sc)
else:
    y_pred_best = baseline_results[best_model_name]['model'].predict(X_test_sc)

residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

axes[0, 0].scatter(y_test, y_pred_best, alpha=0.5, s=25, color='steelblue', edgecolors='white', linewidth=0.3)
lims = [min(y_test.min(), y_pred_best.min()) - 0.1, max(y_test.max(), y_pred_best.max()) + 0.1]
axes[0, 0].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[0, 0].set_xlabel('Actual Ra')
axes[0, 0].set_ylabel('Predicted Ra')
axes[0, 0].set_title(f'Actual vs. Predicted — {best_model_name}')
axes[0, 0].legend()

axes[0, 1].scatter(y_pred_best, residuals, alpha=0.5, s=25, color='steelblue', edgecolors='white', linewidth=0.3)
axes[0, 1].axhline(y=0, color='r', ls='--', lw=1.5)
axes[0, 1].set_xlabel('Predicted Ra')
axes[0, 1].set_ylabel('Residual')
axes[0, 1].set_title('Residuals vs. Predicted')

axes[1, 0].hist(residuals, bins=30, color='steelblue', edgecolor='white', alpha=0.85, density=True)
kde_r = stats.gaussian_kde(residuals)
x_r = np.linspace(residuals.min(), residuals.max(), 200)
axes[1, 0].plot(x_r, kde_r(x_r), 'firebrick', lw=2)
axes[1, 0].axvline(0, color='black', ls='--', lw=1)
axes[1, 0].set_xlabel('Residual')
axes[1, 0].set_title('Residual Distribution')

stats.probplot(residuals, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Residual Q-Q Plot')
axes[1, 1].get_lines()[0].set_markerfacecolor('steelblue')
axes[1, 1].get_lines()[0].set_markersize(4)

plt.tight_layout()
plt.show()

print(f'Residual statistics:')
print(f'  Mean:    {residuals.mean():+.4f}')
print(f'  Std:     {residuals.std():.4f}')
print(f'  Skew:    {stats.skew(residuals):.3f}')
print(f'  Kurt:    {stats.kurtosis(residuals):.3f}')
print(f'  Max |e|: {np.abs(residuals).max():.4f}')

### 16.2 · Error Distribution by Factor Level

Disaggregating prediction errors by each machining parameter level reveals whether the model struggles disproportionately under specific cutting conditions.

In [ ]:
test_analysis = X_test.copy()
test_analysis['abs_error'] = np.abs(y_test.values - y_pred_best)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, col in enumerate(['f', 'vc', 'ap']):
    if col in test_analysis.columns:
        grp = test_analysis.groupby(col)['abs_error'].agg(['mean', 'std', 'count'])
        axes[idx].bar(range(len(grp)), grp['mean'], yerr=grp['std'], capsize=4,
                      color='steelblue', edgecolor='white', alpha=0.8)
        axes[idx].set_xticks(range(len(grp)))
        axes[idx].set_xticklabels([f'{v}' for v in grp.index], fontsize=9)
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Mean Absolute Error')
        axes[idx].set_title(f'MAE by {col} Level')

plt.tight_layout()
plt.show()

### 16.3 · Learning Curve Analysis

Learning curves plot model performance as a function of training set size, providing direct insight into the bias-variance tradeoff.

In [ ]:
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

lc_models = [
    ('Random Forest (tuned)', best_rf),
    ('Gradient Boosting (tuned)', best_gb),
    ('Stacking Ensemble', stacking),
]

for idx, (name, model) in enumerate(lc_models):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train_sc, y_train, cv=5,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='r2', n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)

    axes[idx].fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
    axes[idx].fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='orange')
    axes[idx].plot(train_sizes, train_mean, 'o-', color='blue', label='Training R2', lw=2)
    axes[idx].plot(train_sizes, val_mean, 'o-', color='orange', label='Validation R2', lw=2)
    axes[idx].set_xlabel('Training Set Size')
    axes[idx].set_ylabel('R2 Score')
    axes[idx].set_title(f'Learning Curve — {name}')
    axes[idx].legend(loc='lower right', fontsize=8)
    axes[idx].set_ylim([0.5, 1.05])

plt.tight_layout()
plt.show()

### 16.4 · Cross-Validation Stability Analysis

Repeated K-fold cross-validation (5 folds x 3 repeats = 15 evaluations) produces a robust distribution of R2 scores for each model, allowing comparison of both central tendency and stability.

In [ ]:
from sklearn.model_selection import RepeatedKFold

rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=SEED)

cv_models = {
    'RF (tuned)': best_rf,
    'GB (tuned)': best_gb,
    'Ridge': Ridge(alpha=1.0),
    'Stacking': stacking,
}

cv_distributions = {}
for name, model in cv_models.items():
    scores = cross_val_score(model, X_train_sc, y_train, cv=rkf, scoring='r2', n_jobs=-1)
    cv_distributions[name] = scores

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(cv_distributions.values(), labels=cv_distributions.keys(),
                patch_artist=True, widths=0.5)
palette = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('R2 Score')
ax.set_title('Cross-Validation R2 Distribution (5-Fold x 3 Repeats)')
plt.tight_layout()
plt.show()

print('--- CV Stability Report ---')
for name, scores in cv_distributions.items():
    print(f'  {name:15s}: mean={scores.mean():.4f}  std={scores.std():.4f}  min={scores.min():.4f}  max={scores.max():.4f}')

## 17 · Model Persistence for Factory Deployment

The deployment pipeline requires two serialized artifacts: the trained model and the feature scaler.

In [ ]:
import joblib

deployment_bundle = {
    'model': stacking if best_model_name == 'Stacking Ensemble' else (
        tuned_results[best_model_name]['model'] if best_model_name in tuned_results
        else baseline_results[best_model_name]['model']
    ),
    'scaler': scaler_model,
    'features': X_train_sc.columns.tolist(),
    'best_model_name': best_model_name,
    'test_r2': leaderboard['Test R2'].iloc[0],
    'test_rmse': leaderboard['Test RMSE'].iloc[0],
}

joblib.dump(deployment_bundle, 'Surface_Quality_Predictor.pkl')
print(f'Deployment bundle saved: Surface_Quality_Predictor.pkl')
print(f'  Model:    {best_model_name}')
print(f'  Test R2:  {leaderboard["Test R2"].iloc[0]:.4f}')
print(f'  Features: {len(X_train_sc.columns)}')

### 17.1 · Unscaled Deployment Model for Web Application

For a production web application where operators input raw factory measurements, the model is trained on unscaled data.

In [ ]:
df1_raw = pd.read_csv('Exp1.csv')
df2_raw = pd.read_csv('Exp2.csv')
raw_df = pd.concat([df1_raw, df2_raw], ignore_index=True)
raw_df = raw_df.replace('na', np.nan)

for col in ['TCond', 'vc', 'f', 'ap', 'Ra']:
    raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')
raw_df = raw_df.dropna(subset=['TCond', 'vc', 'f', 'ap', 'Ra'])

X_app = raw_df[['vc', 'f', 'ap', 'TCond']]
y_app = raw_df['Ra']

app_model = RandomForestRegressor(n_estimators=300, max_depth=20, random_state=SEED, n_jobs=-1)
app_model.fit(X_app, y_app)

app_pred = app_model.predict(X_app)
app_r2 = r2_score(y_app, app_pred)
print(f'App model (raw 4-feature RF) — Training R2: {app_r2:.4f}')

joblib.dump(app_model, 'Virtual_Quality_Inspector_App_Model.pkl')
print('Saved: Virtual_Quality_Inspector_App_Model.pkl')

## 18 · Conclusion

This project demonstrates that integrating data-driven modeling into CNC turning operations transforms quality control from a reactive post-process measurement into a proactive pre-process forecast. Key findings:

1. **Feed rate is the dominant roughness driver**, confirming the theoretical Ra proportional to f squared relationship. Mean Ra increases from 0.43 at f=0.07 to 0.98 at f=0.13.

2. **Cutting forces carry additional predictive information** beyond the controllable parameters alone, particularly the radial force Fy which governs tool deflection.

3. **Tool wear (TCond) has a surprisingly weak main effect** in this dataset, likely because the experiments were conducted under cutting fluid conditions that delayed wear mechanisms. However, the interaction between TCond and feed rate reveals that worn tools amplify the roughness penalty at high feed rates.

4. **Ensemble methods consistently outperform individual models**, with the stacking meta-learner achieving the highest holdout R2 by combining complementary model architectures.

5. **Feature engineering rooted in metal-cutting physics** (chip area, MRR, specific energy, force ratios) provides meaningful predictive gains over raw parameters alone.

The resulting Virtual Quality Inspector enables process engineers to receive an instantaneous surface roughness forecast before the machining cycle completes.

## 19 · Future Scope

- **Real-Time Sensor Fusion:** Integrating live acoustic emission and vibration accelerometer signals for in-process quality updates.
- **Multi-Material Expansion:** Retraining on Titanium Ti-6Al-4V, Inconel 718, and other aerospace alloys for a universal surface quality analytics platform.
- **Prescriptive Optimization:** Using the trained model as an objective function for Bayesian optimization of cutting parameters.
- **Edge Computing Deployment:** Embedding the serialized model onto the CNC controller's embedded Linux environment for sub-second predictions.
- **Continual Learning:** Implementing an online learning pipeline that incrementally updates model weights with new measurement data.